# Natura reference sampling

**Inputs**
- KESE forest monitoring CSV: several rows per 20 m-radius monitoring plot.
- Natura habitat inventory polygons: optional, but useful for checking labels
  and (if enabled) creating additional **weakly labelled** training points.

**Outputs**
- Monitoring plot centres (points) and 20 m plot circles (polygons).
- Inventory polygons with a geographic fold assigned to each.
- Five geographic folds: fold 0 = final test; folds 1–4 = rotating CV folds.
- Optional points inside inventory polygons; quality-control tables and maps.

This notebook does **not** fit a model or download Sentinel/LiDAR raster data.
The exact 10 km block and 1 km separation are provisional design decisions.
They do not guarantee statistical independence.

**Before running:** change the paths in the next cell. Run cells from top to
bottom. Python needs pandas, numpy, geopandas, shapely, scipy and scikit-learn.



In [1]:
# ================================================================
# 1. SETTINGS — normally you only need to edit this cell
# ================================================================

from pathlib import Path

# Monitoring file. Each plot may appear on several CSV rows, one per indicator.
MONITORING_CSV = Path(
    r"C:\Users\kaursa40847\project\original\detailed_monitoring_plots_2017_2026_est.csv"
)

# Natura inventory polygons. The script can run without this file.
INVENTORY_FILE = Path(
    r"C:\Users\kaursa40847\project\working\natura_habitatsPolygon_forests.gpkg"
)

# Name of the polygon layer inside the GPKG. Use None for its default layer.
INVENTORY_LAYER = "natura_habitatsPolygon_forests"

# A dedicated folder: outputs with the same names are replaced when rerunning.
OUTPUT_DIR = Path(r"C:\Users\kaursa40847\project\working\natura_reference_output_v2")

# Years used for reporting. Records outside these years are NOT automatically
# removed, because the original notebook keeps them and flags them for QC.
YEAR_WINDOW = (2017, 2026)

# All processing is done in L-EST97, where map coordinates are in metres.
CRS = "EPSG:3301"

# Geographic block size: 10,000 metres = 10 km square blocks.
SPATIAL_BLOCK_M = 10_000

# There are five folds: 0 is the untouched final TEST; 1,2,3,4 are used
# for training and rotating validation during later model tuning.
N_FOLDS = 5
TEST_FOLD = 0

# The field measurement used to approximately balance folds.
# Possible values: habitat_type, structural_preservation,
# overall_conservation_value. This does not guarantee perfect class balance.
BALANCE_FIELD = "structural_preservation"

# Training records closer than 1 km to a test record are flagged and excluded.
# The same distance is later used between CV training and validation records.
TEST_BUFFER_M = 1_000

# Randomness stays reproducible, i.e. reruns choose the same folds/points.
SEED = 42

# False = only use independent field plots for the initial model (recommended).
# True  = also generate points inside polygon inventories as OPTIONAL weak labels.
MAKE_POLYGON_SAMPLES = False

# Settings used only if MAKE_POLYGON_SAMPLES is True.
MAX_SAMPLES_PER_POLYGON = 6
INNER_BUFFER_M = 20       # Keep sampled points away from polygon boundaries.
MIN_POINT_DISTANCE_M = 60  # Spread points apart WITHIN each polygon.



In [2]:
# ================================================================
# 2. IMPORTS AND FIELD-NAME TRANSLATIONS
# ================================================================

import hashlib
import math
import re
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from shapely.geometry import Point, Polygon, MultiPolygon, box
from shapely.validation import make_valid
from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_columns", 30)

# A Natura code with an asterisk means a priority habitat. Some raw sources
# leave off the asterisk; normalize those types consistently.
STARRED_TYPES = {"9010", "9020", "9080", "9180", "91D0", "91E0"}

# In the monitoring CSV, the long Estonian name describes what was measured.
# The right-hand name is the short column name we will use in modelling.
PLOT_COLUMNS = {
    "Elupaigatüüp (LD)": "habitat_type",
    "Elupaiga esinduslikkus": "representativeness",
    "Koosluse struktuuri säilimine": "structural_preservation",
    "Elupaiga funktsioonide säilimine": "functional_preservation",
    "Elupaiga üldine looduskaitseline väärtus": "overall_conservation_value",
    "Kasvukohatüüp (Paal 1997)": "growth_site_type",
}

# The same idea, but for fields in the Natura inventory GeoPackage.
# Change the values on the right if your inventory's field names differ.
INVENTORY_FIELDS = {
    "polygon_id": "id",
    "habitat_type": "pohityyp",
    "representativeness": "esinduslik",
    "structural_preservation": "str_sail",
    "functional_preservation": "funkt_sail",
    "restorability": "taastatavs",
    "conservation_status": "lk_seisund",
    "overall_conservation_value": "uldine_lk",
    "inventory_date": "invent_kpv",
    "protected_area": "kaitst_ala",
}




## 3. Small helper functions

These functions handle repeated jobs such as cleaning codes and assigning
10 km block names. **They do not yet read or change any input files.**



In [3]:
def habitat_code(value):
    """Convert habitat text such as '9010* ...' into a short Natura code."""

    # Missing values must stay missing; do not turn them into a new habitat.
    if pd.isna(value):
        return None

    # Look only at the beginning of the response for a 4-character habitat
    # code. '0' is kept if the source says there is no habitat of this type.
    text = str(value).strip()
    match = re.match(r"^(?:Eesti\s+)?(0|\d[0-9A-Z]{3}\*?)", text, re.I)

    if match is None:
        return None

    # E.g. '91D0*' becomes '91D0', and the asterisk is then standardized.
    code = match.group(1).upper().rstrip("*")
    if code in STARRED_TYPES:
        return code + "*"
    return code


def rating_code(value):
    """Keep the initial rating letter (A, B, C, D, P) or Roman numeral."""

    if pd.isna(value):
        return None

    text = str(value).strip().upper()
    # Examples: 'B - hea' -> B; 'III: ...' -> III.
    match = re.match(r"^(IV|III|II|I|[A-DP])(?=\s|[-.:]|$)", text)
    if match is None:
        return None
    return match.group(1)


def date_to_year(series):
    """Interpret Estonian dates and return the year for each row."""

    dates = pd.to_datetime(series, errors="coerce", dayfirst=True, format="mixed")
    return dates.dt.year


def block_name(point):
    """Name the 10 km x 10 km square containing this location."""

    # Divide a metre-coordinate by 10,000 and round DOWN to identify its square.
    x_number = math.floor(point.x / SPATIAL_BLOCK_M)
    y_number = math.floor(point.y / SPATIAL_BLOCK_M)
    return f"B{x_number}_{y_number}"


def stable_number(value):
    """Create a repeatable integer seed from a polygon or block identifier."""

    # Python's built-in hash() can change between sessions. SHA-256 does not.
    string = f"{SEED}:{value}"
    hash_text = hashlib.sha256(string.encode("utf-8")).hexdigest()
    return int(hash_text[:16], 16)


def save_gpkg(frame, filename, layer_name):
    """Write one vector dataset without changing its input source."""

    if frame is None or len(frame) == 0:
        print("Skipped empty output:", filename)
        return

    output_path = OUTPUT_DIR / filename
    frame.to_file(output_path, layer=layer_name, driver="GPKG", index=False)




## 4. Read monitoring observations

A single circular field plot has several CSV rows, because each row stores
a different observation, such as habitat type or structural preservation.
We turn this **long table** into **one row per monitored plot**.

**Special coordinate warning:** this CSV has the usual easting and northing
values under unusual column labels: `Sisestatud y L-EST97` is easting, and
`Sisestatud x L-EST97` is northing. The swap is deliberate for this dataset.



In [4]:
def prepare_monitoring():
    """Return field-plot centres, bad-coordinate rows, and indicator conflicts."""

    # Step 1: read each value as text, because values include letters and codes.
    if not MONITORING_CSV.exists():
        raise FileNotFoundError(f"Cannot find monitoring CSV: {MONITORING_CSV}")

    raw = pd.read_csv(MONITORING_CSV, sep=";", dtype=str, encoding="utf-8-sig")

    # Step 2: keep only forest-habitat monitoring records.
    is_forest = raw["Seiretöö nimetus"].str.startswith(
        "Metsaelupaikade seire", na=False
    )
    forest = raw.loc[is_forest].copy()

    # Step 3: a plot must have an observation ID to combine its indicator rows.
    observation_field = "Proovi / vaatluse kood"
    indicator_field = "Näitaja nimetus"
    value_field = "Väärtus (muud)"
    forest = forest.dropna(subset=[observation_field])

    # Step 4: limit observations to the r=20 m field protocol.
    # Other protocols must not silently be treated as 20 m circles.
    radius_text = forest["Vaatlusgrupp"].str.extract(
        r"r\s*=\s*(\d+)m", expand=False
    )
    correct_radius = radius_text.eq("20")
    if (~correct_radius).any():
        n_skipped = int((~correct_radius).sum())
        warnings.warn(f"Skipped {n_skipped} rows without explicit r=20m protocol.")
    forest = forest.loc[correct_radius].copy()

    # Step 5: find observations where one indicator was entered more than once
    # with different answers. Export them for manual inspection.
    distinct_values = forest.groupby(
        [observation_field, indicator_field]
    )[value_field].nunique(dropna=True)
    conflicts = distinct_values[distinct_values > 1].reset_index()
    conflicts = conflicts.rename(columns={value_field: "n_distinct_values"})
    if len(conflicts):
        warnings.warn(f"Found {len(conflicts)} plot-indicator conflicts; first value is used.")

    # Step 6: keep location/date information once per observation.
    location_fields = [
        "Seirekoha nimi",
        "Seirekogumi nimi",
        "Seireaja algus",
        "Sisestatud x L-EST97",
        "Sisestatud y L-EST97",
    ]
    plot_info = forest.groupby(observation_field, as_index=False)[location_fields].first()

    # Step 7: make one column per indicator (pivoting long data to wide data).
    # aggfunc='first' resolves repeated entries; conflicts were recorded above.
    indicators = forest.pivot_table(
        index=observation_field,
        columns=indicator_field,
        values=value_field,
        aggfunc="first",
    ).reset_index()
    wide = plot_info.merge(indicators, on=observation_field, how="left", validate="one_to_one")

    # Step 8: create a simple table with one row for each plot.
    # Do not use the monitoring programme name as the Natura habitat label!
    plots = pd.DataFrame()
    plots["plot_id"] = wide[observation_field].astype(str)
    plots["location_name"] = wide["Seirekoha nimi"]
    plots["monitoring_group"] = wide["Seirekogumi nimi"]
    plots["inventory_date"] = wide["Seireaja algus"]

    # Step 9: convert text coordinates into numbers.
    # The CSV's x/y labels are reversed relative to the usual GIS convention.
    east_text = wide["Sisestatud y L-EST97"].str.replace(",", ".", regex=False)
    north_text = wide["Sisestatud x L-EST97"].str.replace(",", ".", regex=False)
    plots["easting"] = pd.to_numeric(east_text, errors="coerce")
    plots["northing"] = pd.to_numeric(north_text, errors="coerce")

    # Step 10: translate original indicator names into consistent model fields.
    # Store both raw and cleaned values so that you can check interpretations.
    for original_field, model_field in PLOT_COLUMNS.items():
        raw_field = "raw_" + model_field

        if original_field in wide.columns:
            plots[raw_field] = wide[original_field]
        else:
            plots[raw_field] = None

        if model_field == "habitat_type":
            plots[model_field] = plots[raw_field].map(habitat_code)
        elif model_field == "growth_site_type":
            plots[model_field] = plots[raw_field]
        else:
            plots[model_field] = plots[raw_field].map(rating_code)

    # Step 11: add basic metadata for later modelling and quality checks.
    plots["inventory_year"] = date_to_year(plots["inventory_date"])
    plots["in_year_window"] = plots["inventory_year"].between(*YEAR_WINDOW).fillna(False)
    plots["plot_radius_m"] = 20
    plots["reference_source"] = "field_plot"
    plots["label_scope"] = "20m_plot"

    # Step 12: reject points with clearly impossible Estonian coordinates.
    # This is a broad plausibility test, not an accuracy guarantee.
    valid_east = plots["easting"].between(300_000, 800_000)
    valid_north = plots["northing"].between(6_300_000, 6_700_000)
    valid_coordinates = valid_east & valid_north
    rejected = plots.loc[~valid_coordinates].copy()
    plots = plots.loc[valid_coordinates].copy().reset_index(drop=True)

    # Step 13: turn coordinate columns into actual GIS point geometries.
    points = gpd.points_from_xy(plots["easting"], plots["northing"])
    centres = gpd.GeoDataFrame(plots, geometry=points, crs=CRS)

    # Step 14: assign each centre to its geographic 10 km block.
    centres["block_id"] = centres.geometry.map(block_name)

    print("Monitoring plots:", len(centres))
    print("Rejected coordinates:", len(rejected))
    print("Non-missing target values:")
    print(centres[["habitat_type", "structural_preservation", "overall_conservation_value"]].count())
    return centres, rejected, conflicts




## 5. Read inventory polygons (optional)

Geometry is repaired when possible and kept in L-EST97. An inventory
assessment describes a **whole polygon**, even if that polygon contains
young, old or disturbed patches. This label is not a verified point label.

The original GeoPackage is only read, never edited.



In [5]:
def keep_polygon_parts(geometry):
    """Repair a geometry and return only its polygon pieces, if any."""

    if geometry is None or geometry.is_empty:
        return None

    if not geometry.is_valid:
        geometry = make_valid(geometry)

    if isinstance(geometry, (Polygon, MultiPolygon)):
        return geometry if not geometry.is_empty else None

    # Geometry repair can sometimes produce a GeometryCollection containing
    # both lines and polygons. Only polygon pieces represent habitat area.
    polygon_parts = []
    if hasattr(geometry, "geoms"):
        for part in geometry.geoms:
            if isinstance(part, Polygon) and not part.is_empty:
                polygon_parts.append(part)

    if len(polygon_parts) == 0:
        return None
    return MultiPolygon(polygon_parts)


def prepare_inventory():
    """Read Natura polygons, clean their ratings and attach identifiers."""

    if not INVENTORY_FILE.exists():
        warnings.warn(f"Inventory missing; continuing with plots only: {INVENTORY_FILE}")
        return None

    # The layer argument is needed only for GeoPackages with named layers.
    if INVENTORY_LAYER is None:
        raw = gpd.read_file(INVENTORY_FILE)
    else:
        raw = gpd.read_file(INVENTORY_FILE, layer=INVENTORY_LAYER)

    # Project to a metre-based CRS so buffer and area calculations make sense.
    if raw.crs is None:
        raise ValueError("Inventory CRS is unknown. Define it correctly in QGIS first.")
    raw = raw.to_crs(CRS).reset_index(drop=True)

    # Start a separate table of standardized fields, with the same row order.
    result = pd.DataFrame(index=raw.index)
    for model_field, original_field in INVENTORY_FIELDS.items():
        if original_field in raw.columns:
            result[model_field] = raw[original_field]
        else:
            result[model_field] = None
            warnings.warn(f"Missing inventory field: {original_field}")

    # polygon_row_id is a guaranteed UNIQUE row identifier for this run.
    # polygon_id retains the external ID when available; it might not be unique.
    result["polygon_row_id"] = np.arange(len(raw))
    result["polygon_id"] = result["polygon_id"].astype("string")
    no_id = result["polygon_id"].isna()
    result.loc[no_id, "polygon_id"] = [
        f"source_row_{row_number}" for row_number in result.index[no_id]
    ]

    # Preserve raw ratings, and create short ratings (A/B/C...) for analysis.
    rating_fields = [
        "representativeness", "structural_preservation", "functional_preservation",
        "conservation_status", "overall_conservation_value",
    ]
    for field in rating_fields:
        result["raw_" + field] = result[field]
        result[field] = result[field].map(rating_code)

    result["raw_habitat_type"] = result["habitat_type"]
    result["habitat_type"] = result["habitat_type"].map(habitat_code)
    result["inventory_year"] = date_to_year(result["inventory_date"])
    result["in_year_window"] = result["inventory_year"].between(*YEAR_WINDOW).fillna(False)
    result["reference_source"] = "inventory_polygon"
    result["label_scope"] = "whole_polygon"

    # Turn the attribute table back into a GIS layer with repaired geometries.
    repaired_shapes = raw.geometry.map(keep_polygon_parts)
    inventory = gpd.GeoDataFrame(result, geometry=repaired_shapes, crs=CRS)

    # Remove irreparable and zero-area polygons. This does not edit the source.
    invalid_shape = inventory.geometry.isna() | inventory.geometry.is_empty
    if invalid_shape.any():
        warnings.warn(f"Excluded {int(invalid_shape.sum())} unusable geometries.")
    inventory = inventory.loc[~invalid_shape].copy()
    inventory["area_ha"] = inventory.geometry.area / 10_000
    inventory = inventory.loc[inventory["area_ha"] > 0].copy().reset_index(drop=True)

    # Inventory polygons initially inherit their block from an interior point.
    # Some will later inherit a matched monitoring plot's block instead.
    interior_points = inventory.geometry.representative_point()
    inventory["block_id"] = interior_points.map(block_name)

    print("Inventory polygons:", len(inventory))
    print("Total polygon area, ha:", round(inventory["area_ha"].sum(), 1))
    return inventory




## 6. Assign geographical folds and guard against plot–polygon leakage

1. Divide field-plot centres into 10 km blocks.
2. Assign **whole blocks** to one of five folds, approximately balancing the
   selected target label (`BALANCE_FIELD`).
3. Make fold 0 the final test; folds 1–4 will rotate as validation during
   **later** model training. No permanent separate validation fold is needed.
4. An inventory polygon intersecting a field centre inherits that field
   plot's fold. If it intersects plots assigned to conflicting folds/blocks,
   those records are flagged rather than being used across data partitions.

The point-in-polygon match uses a plot **centre**, not its whole 20 m circle.
Such matching should not be read as evidence that the entire field plot lies
inside the inventory polygon.



In [6]:
def assign_geographic_folds(centres, inventory):
    """Assign five folds, with all centres in a geographic block kept together."""

    n_blocks = centres["block_id"].nunique()
    if n_blocks < N_FOLDS:
        raise ValueError(f"Only {n_blocks} distinct geographic blocks; need {N_FOLDS}.")

    # Missing target labels form their own 'UNKNOWN' balance group.
    classes = centres[BALANCE_FIELD].fillna("UNKNOWN").astype(str)

    # StratifiedGroupKFold tries to distribute target classes over folds while
    # NEVER splitting one geographic block between folds.
    splitter = StratifiedGroupKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED,
    )

    # Build a dictionary such as {'B55_645': 0, 'B56_645': 3, ...}.
    fold_by_block = {}
    for fold_number, (_training_rows, held_out_rows) in enumerate(
        splitter.split(centres, classes, groups=centres["block_id"])
    ):
        held_out_blocks = centres.iloc[held_out_rows]["block_id"].unique()
        for block in held_out_blocks:
            fold_by_block[block] = fold_number

    centres = centres.copy()
    centres["fold_id"] = centres["block_id"].map(fold_by_block).astype(int)
    centres["split"] = "train_cv"
    centres.loc[centres["fold_id"] == TEST_FOLD, "split"] = "test"

    # An inventory file is optional: field-only workflows can stop here.
    if inventory is None:
        return centres, None, pd.DataFrame()

    inventory = inventory.copy()

    # Some polygons occur in blocks containing no monitoring plots. We must
    # still assign them a REPEATABLE fold, without looking at model scores.
    missing_blocks = inventory.loc[
        ~inventory["block_id"].isin(fold_by_block), "block_id"
    ].unique()
    for block in sorted(missing_blocks):
        fold_by_block[block] = stable_number(block) % N_FOLDS

    inventory["fold_id"] = inventory["block_id"].map(fold_by_block).astype(int)
    inventory["split"] = "train_cv"
    inventory.loc[inventory["fold_id"] == TEST_FOLD, "split"] = "test"

    # Join a field plot to all polygons containing its centre.
    # Matching is only for quality-control and safe fold assignment.
    compared_fields = [
        "habitat_type", "structural_preservation",
        "overall_conservation_value", "inventory_year",
    ]
    left_fields = ["plot_id", "block_id", "fold_id", *compared_fields, "geometry"]
    right_fields = ["polygon_row_id", "polygon_id", *compared_fields, "geometry"]

    matching = gpd.sjoin(
        centres[left_fields],
        inventory[right_fields],
        how="inner",
        predicate="intersects",
    )

    # Keep this comparison table for checking disagreement in habitat/condition.
    links = matching.drop(columns="geometry", errors="ignore").copy()
    links = links.rename(columns={
        "habitat_type_left": "plot_habitat_type",
        "habitat_type_right": "inventory_habitat_type",
    })

    # A comparison is meaningful only when BOTH labels are present.
    for field in ["habitat_type", "structural_preservation", "overall_conservation_value"]:
        if field == "habitat_type":
            plot_field = "plot_habitat_type"
            inventory_field = "inventory_habitat_type"
        else:
            plot_field = field + "_left"
            inventory_field = field + "_right"

        both_present = links[plot_field].notna() & links[inventory_field].notna()
        links[field + "_agrees"] = pd.NA
        links.loc[both_present, field + "_agrees"] = (
            links.loc[both_present, plot_field]
            == links.loc[both_present, inventory_field]
        )

    links["inventory_year_difference"] = (
        pd.to_numeric(links["inventory_year_left"], errors="coerce")
        - pd.to_numeric(links["inventory_year_right"], errors="coerce")
    )

    # One polygon might contain several field-plot centres.
    # They must not make that polygon appear in DIFFERENT folds.
    conflicting_polygons = set()
    conflicting_plots = set()

    for polygon_row_id, plot_matches in matching.groupby("polygon_row_id"):
        matched_folds = plot_matches["fold_id"].unique()
        matched_blocks = plot_matches["block_id"].unique()

        if len(matched_folds) == 1 and len(matched_blocks) == 1:
            # Safe case: ALL matching plots agree on both fold and block.
            polygon_mask = inventory["polygon_row_id"] == polygon_row_id
            chosen_fold = int(matched_folds[0])
            inventory.loc[polygon_mask, "fold_id"] = chosen_fold
            inventory.loc[polygon_mask, "block_id"] = matched_blocks[0]

            if chosen_fold == TEST_FOLD:
                inventory.loc[polygon_mask, "split"] = "test"
            else:
                inventory.loc[polygon_mask, "split"] = "train_cv"
        else:
            # Unsafe case: keep records for checking, but flag them as excluded.
            conflicting_polygons.add(polygon_row_id)
            conflicting_plots.update(plot_matches["plot_id"])

    inventory.loc[
        inventory["polygon_row_id"].isin(conflicting_polygons), "split"
    ] = "overlap_excluded"
    centres.loc[
        centres["plot_id"].isin(conflicting_plots), "split"
    ] = "overlap_excluded"

    if conflicting_polygons:
        warnings.warn(
            f"Flagged {len(conflicting_polygons)} cross-fold polygons "
            f"and {len(conflicting_plots)} field plots."
        )

    print("Plot-polygon matches:", len(links))
    print("Distinct matched field plots:", matching["plot_id"].nunique())
    return centres, inventory, links




## 7. Optional points inside Natura inventory polygons

Turn `MAKE_POLYGON_SAMPLES = True` only when you want **additional weak
labels**. These points may have different local raster features but will all
inherit the parent polygon's habitat/condition assessment.

Their `sample_weight` sums
to 1 within a polygon, preventing large polygons with many chosen points from
automatically receiving more weight simply due to point count.



In [7]:
def points_inside_polygon(polygon, wanted_count, polygon_key):
    """Choose separated points inside one polygon, when its shape allows it."""

    # Make a smaller copy of the polygon to avoid sampling close to the border.
    interior = polygon.buffer(-INNER_BUFFER_M)

    # Narrow polygons might vanish after buffering. Use one safe interior point.
    if interior.is_empty or interior.area <= 0:
        return [polygon.representative_point()], True

    # The seed makes random point locations repeatable across runs.
    generator = np.random.default_rng(stable_number(polygon_key))
    xmin, ymin, xmax, ymax = interior.bounds

    # Start with one point that Shapely guarantees lies inside the geometry.
    chosen = [interior.representative_point()]

    # Try at most 200 random candidates; long, narrow polygons can be difficult.
    for attempt in range(200):
        if len(chosen) >= wanted_count:
            break

        x = generator.uniform(xmin, xmax)
        y = generator.uniform(ymin, ymax)
        candidate = Point(x, y)

        if not interior.contains(candidate):
            continue

        # Reject points too close to points already selected in this polygon.
        far_enough = True
        for earlier_point in chosen:
            if candidate.distance(earlier_point) < MIN_POINT_DISTANCE_M:
                far_enough = False
                break

        if far_enough:
            chosen.append(candidate)

    return chosen, False


def sample_inventory(inventory):
    """Make optional weakly labelled points while preserving polygon fold IDs."""

    if inventory is None or not MAKE_POLYGON_SAMPLES:
        return None

    copied_fields = [
        "polygon_row_id", "polygon_id", "habitat_type", "representativeness",
        "structural_preservation", "functional_preservation", "restorability",
        "conservation_status", "overall_conservation_value", "inventory_year",
        "inventory_date", "area_ha", "block_id", "fold_id", "split",
        "protected_area",
    ]
    rows = []

    # itertuples() processes many inventory polygons faster than iterrows().
    for polygon in inventory.itertuples(index=False):
        if polygon.split == "overlap_excluded":
            continue

        # More area can give more points, but never more than the configured cap.
        scaled_count = math.ceil(1.5 * math.sqrt(polygon.area_ha))
        wanted_count = min(MAX_SAMPLES_PER_POLYGON, max(1, scaled_count))

        points, used_edge_fallback = points_inside_polygon(
            polygon.geometry,
            wanted_count,
            polygon.polygon_row_id,
        )

        # All sampled points inherit the SAME polygon-level reference labels.
        # They also inherit the SAME fold, avoiding point-by-point leakage.
        attributes = {}
        for field in copied_fields:
            attributes[field] = getattr(polygon, field)

        attributes["reference_source"] = "inventory_weak"
        attributes["label_scope"] = "whole_polygon_inherited"
        attributes["edge_fallback"] = used_edge_fallback
        attributes["n_points_in_polygon"] = len(points)
        attributes["sample_weight"] = 1 / len(points)

        for point_number, point in enumerate(points, start=1):
            one_row = attributes.copy()
            one_row["sample_id"] = f"inv_{polygon.polygon_row_id}_{point_number}"
            one_row["geometry"] = point
            rows.append(one_row)

    if len(rows) == 0:
        print("No eligible inventory polygons for point sampling.")
        return None

    samples = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS)
    print("Weak-label sample points:", len(samples))
    print("Source inventory polygons:", samples["polygon_row_id"].nunique())
    return samples




## 8. Keep training observations away from test / validation observations

Two separate safeguards are used:
- **Now:** mark training observations within 1 km of final test points as
  `buffer_excluded`. They will not be used to fit a model.
- **Later, during modelling:** `buffered_cv_indices()` temporarily excludes
  nearby training observations for each rotating validation fold.

These buffers are calculated using **centre-to-centre distance**, not the
distance between the 20 m plot boundaries. A 1 km buffer is provisional.



In [8]:
def exclude_near_test(centres, samples=None, buffer_m=TEST_BUFFER_M):
    """Flag training centres too close to held-out final test centres."""

    # Always include monitoring centres; optionally include inventory points.
    datasets = [centres]
    if samples is not None and len(samples) > 0:
        datasets.append(samples)

    # Collect all points reserved for final testing, across the two sources.
    test_points = []
    for dataset in datasets:
        subset = dataset.loc[dataset["split"] == "test"]
        test_points.extend(list(subset.geometry))

    if len(test_points) == 0:
        raise ValueError("There are no eligible test samples after overlap checks.")

    # cKDTree finds the nearest test point efficiently (faster than every pair).
    test_coordinates = np.array([[point.x, point.y] for point in test_points])
    test_tree = cKDTree(test_coordinates)

    processed = []
    for dataset in datasets:
        updated = dataset.copy()
        training_mask = updated["split"] == "train_cv"
        candidate_points = updated.loc[training_mask, "geometry"]

        if len(candidate_points) > 0:
            xy = np.array([[point.x, point.y] for point in candidate_points])
            distances, _nearest_ids = test_tree.query(xy, k=1)
            too_near = distances < buffer_m

            # Keep rows in the file but mark them as NOT eligible for modelling.
            excluded_rows = candidate_points.index[too_near]
            updated.loc[excluded_rows, "split"] = "buffer_excluded"

        processed.append(updated)

    updated_centres = processed[0]
    updated_samples = processed[1] if len(processed) == 2 else None
    return updated_centres, updated_samples


def buffered_cv_indices(samples, validation_fold, buffer_m=TEST_BUFFER_M):
    """Return row indices for one spatial training/validation round.

    Call this LATER, after raster features have been attached and unusable rows
    removed. For scikit-learn, reset the modelling DataFrame index to 0..n-1.
    Never include `split == 'test'` in the supplied tuning dataset.
    """

    # Start from eligible training/CV samples only; exclude test and QC records.
    eligible = samples.loc[samples["split"] == "train_cv"]

    # This fold is validation; the other folds are candidates for model fitting.
    validation = eligible.loc[eligible["fold_id"] == validation_fold]
    training = eligible.loc[eligible["fold_id"] != validation_fold]

    if len(validation) == 0 or len(training) == 0:
        raise ValueError("This CV round has no training or validation samples.")

    if buffer_m > 0:
        validation_xy = np.array([[p.x, p.y] for p in validation.geometry])
        training_xy = np.array([[p.x, p.y] for p in training.geometry])

        # Exclude training samples whose nearest validation centre is <1 km.
        validation_tree = cKDTree(validation_xy)
        distances, _ = validation_tree.query(training_xy, k=1)
        training = training.iloc[np.where(distances >= buffer_m)[0]]

    # For GridSearchCV, these must be POSITIONAL row indices (0,1,2,...).
    return training.index.to_numpy(), validation.index.to_numpy()




## 9. Save results and quality-control tables

The script saves both plot centres and plot **circles**. The 20 m circles are
used later when collecting Sentinel/LiDAR values inside a measured plot.
Any extra 50 m or 150 m **neighbourhood** features can be computed later with
raster moving windows.



In [9]:
def export_results(centres, inventory, samples, links, rejected, conflicts):
    """Export GIS layers and small CSV tables that describe the reference data."""

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # One record/point per independent monitoring plot.
    save_gpkg(centres, "monitoring_centres.gpkg", "monitoring_centres")

    # A circle of radius 20 m around the same point, preserving its attributes.
    circles = centres.copy()
    circles.geometry = circles.geometry.buffer(20, resolution=24)
    save_gpkg(circles, "monitoring_plots_20m.gpkg", "monitoring_plots_20m")

    # Inventory polygons include the assigned fold and all QC exclusions.
    if inventory is not None:
        save_gpkg(inventory, "inventory_polygons_split.gpkg", "inventory_polygons")

    # Only generated if MAKE_POLYGON_SAMPLES=True.
    if samples is not None and len(samples) > 0:
        save_gpkg(samples, "inventory_sample_points.gpkg", "inventory_samples")

    # Diagnostic tables: overlap/mismatched labels, invalid coordinates,
    # and indicators that were entered inconsistently.
    if len(links) > 0:
        links.to_csv(OUTPUT_DIR / "monitoring_inventory_links.csv", index=False)
    rejected.to_csv(OUTPUT_DIR / "monitoring_invalid_coordinates.csv", index=False)
    conflicts.to_csv(OUTPUT_DIR / "monitoring_conflicting_indicators.csv", index=False)

    # Create a 10 km map grid showing which geographical squares belong to
    # which fold. This is particularly useful in QGIS for visual inspection.
    group_tables = [centres[["block_id", "fold_id"]]]
    if inventory is not None:
        group_tables.append(inventory[["block_id", "fold_id"]])

    groups = pd.concat(group_tables, ignore_index=True).drop_duplicates()

    # Linked polygons may inherit another block's fold; in that case the same
    # square can appear twice. Keep and WARN rather than hiding the mismatch.
    if groups["block_id"].duplicated().any():
        warnings.warn("Some blocks have multiple folds after plot-polygon linking.")

    square_rows = []
    for group in groups.itertuples(index=False):
        # E.g. 'B55_645' means x=55*10km and y=645*10km.
        text = group.block_id.removeprefix("B")
        east_number, north_number = map(int, text.split("_"))
        xmin = east_number * SPATIAL_BLOCK_M
        ymin = north_number * SPATIAL_BLOCK_M
        square = box(xmin, ymin, xmin + SPATIAL_BLOCK_M, ymin + SPATIAL_BLOCK_M)

        square_rows.append({
            "block_id": group.block_id,
            "fold_id": group.fold_id,
            "geometry": square,
        })

    geographic_blocks = gpd.GeoDataFrame(square_rows, geometry="geometry", crs=CRS)
    save_gpkg(geographic_blocks, "spatial_groups.gpkg", "spatial_groups")

    # Summarize the number of observations and known labels per source/fold.
    datasets = [centres]
    if samples is not None:
        datasets.append(samples)

    no_geometry = []
    for dataset in datasets:
        no_geometry.append(dataset.drop(columns="geometry"))
    all_observations = pd.concat(no_geometry, ignore_index=True)

    summary = all_observations.groupby(
        ["reference_source", "split", "fold_id"], dropna=False
    ).agg(
        n_records=("reference_source", "size"),
        n_known_habitat=("habitat_type", "count"),
        n_known_structure=("structural_preservation", "count"),
        n_known_overall_value=("overall_conservation_value", "count"),
        n_blocks=("block_id", "nunique"),
    ).reset_index()
    summary.to_csv(OUTPUT_DIR / "reference_split_summary.csv", index=False)

    # Inspect class balance before accepting the final hold-out split.
    target_fields = [
        "habitat_type", "structural_preservation", "overall_conservation_value"
    ]
    for target in target_fields:
        table = pd.crosstab(centres[target].fillna("UNKNOWN"), centres["fold_id"])
        table.to_csv(OUTPUT_DIR / f"plot_{target}_by_fold.csv")

    # Check whether monitoring years vary between test and training records.
    year_counts = pd.crosstab(
        centres["inventory_year"].fillna(-1), centres["split"]
    )
    year_counts.to_csv(OUTPUT_DIR / "monitoring_year_by_split.csv")

    # Report polygon counts AND areas; they answer different questions.
    if inventory is not None:
        habitat_areas = inventory.groupby("habitat_type", dropna=False).agg(
            n_polygons=("polygon_row_id", "size"),
            area_ha=("area_ha", "sum"),
        )
        habitat_areas.to_csv(OUTPUT_DIR / "inventory_habitat_distribution.csv")

    print("\nMonitoring observations by data split:")
    print(centres.groupby(["split", "fold_id"]).size())
    print("\nAll files saved in:", OUTPUT_DIR)
    return summary




## 10. RUN THE WORKFLOW

**Run this cell after all previous code cells.** Each returned variable is
kept in memory, so you can inspect it in the notebook. The original CSV and
inventory GeoPackage are not modified.



In [18]:
# 1. Read monitoring records and turn them into measured 20 m plot centres.
monitoring_centres, rejected_coordinates, indicator_conflicts = prepare_monitoring()

# 2. Read Natura inventory polygons (or continue with plots only if missing).
inventory_polygons = prepare_inventory()

# 3. Assign geographically grouped folds and check plot-polygon intersections.
monitoring_centres, inventory_polygons, plot_polygon_links = assign_geographic_folds(
    monitoring_centres,
    inventory_polygons,
)

# 4. Optionally create weakly labelled points from inventory polygons.
#    With the default MAKE_POLYGON_SAMPLES=False, this returns None.
inventory_samples = sample_inventory(inventory_polygons)

# 5. Flag training points that lie closer than the buffer to final test points.
monitoring_centres, inventory_samples = exclude_near_test(
    monitoring_centres,
    inventory_samples,
    buffer_m=TEST_BUFFER_M,
)

# 6. Write the GIS layers, class distributions and quality-control tables.
reference_summary = export_results(
    monitoring_centres,
    inventory_polygons,
    inventory_samples,
    plot_polygon_links,
    rejected_coordinates,
    indicator_conflicts,
)

# 7. Show the main summary inside Jupyter.
display(reference_summary)




C:\Users\kaursa40847\AppData\Local\Temp\ipykernel_3640\1324094143.py:41: UserWarning: Found 1 plot-indicator conflicts; first value is used.
  warnings.warn(f"Found {len(conflicts)} plot-indicator conflicts; first value is used.")


Monitoring plots: 1698
Rejected coordinates: 0
Non-missing target values:
habitat_type                  1694
structural_preservation       1260
overall_conservation_value    1482
dtype: int64
Inventory polygons: 20537
Total polygon area, ha: 106689.9


C:\Users\kaursa40847\AppData\Local\Temp\ipykernel_3640\2925774064.py:128: UserWarning: Flagged 3 cross-fold polygons and 12 field plots.
  warnings.warn(
C:\Users\kaursa40847\AppData\Local\Temp\ipykernel_3640\1878426516.py:11: DeprecationWarning: The `resolution` argument to `buffer` is deprecated, `quad_segs` should be used instead to align with shapely.
  circles.geometry = circles.geometry.buffer(20, resolution=24)


Plot-polygon matches: 591
Distinct matched field plots: 590

Monitoring observations by data split:
split             fold_id
buffer_excluded   1            7
                  2            7
                  3            1
overlap_excluded  0            1
                  1            5
                  2            6
test              0          337
train_cv          1          328
                  2          327
                  3          339
                  4          340
dtype: int64

All files saved in: C:\Users\kaursa40847\project\working\natura_reference_output_v2


,reference_source,split,fold_id,n_records,n_known_habitat,n_known_structure,n_known_overall_value,n_blocks
0,field_plot,buffer_excluded,1,7,7,3,7,6
1,field_plot,buffer_excluded,2,7,7,4,6,4
2,field_plot,buffer_excluded,3,1,1,0,1,1
3,field_plot,overlap_excluded,0,1,1,0,1,1
4,field_plot,overlap_excluded,1,5,5,1,5,2
5,field_plot,overlap_excluded,2,6,6,5,6,3
6,field_plot,test,0,337,336,251,298,56
7,field_plot,train_cv,1,328,327,248,280,68
8,field_plot,train_cv,2,327,327,244,287,68
9,field_plot,train_cv,3,339,338,252,295,70


Checking the representativeness of the folds

In [19]:
# Only observations eligible for modelling
data = monitoring_centres[
    monitoring_centres["split"].isin(["train_cv", "test"])
]

# Number of observations and geographic blocks in each fold
print(data.groupby("fold_id").agg(
    n_samples=("plot_id", "size"),
    n_blocks=("block_id", "nunique")
))

# Distribution of target classes across folds
print(pd.crosstab(
    data["structural_preservation"],
    data["fold_id"]
))

         n_samples  n_blocks
fold_id                     
0              337        56
1              328        68
2              327        68
3              339        70
4              340        59
fold_id                    0    1   2   3    4
structural_preservation                       
I                         68   66  65  68   68
II                       100  101  97  99  100
III                       83   81  82  85   84


In [20]:
# Only observations eligible for modelling
data = monitoring_centres[
    monitoring_centres["split"].isin(["train_cv", "test"])
]

# Number of observations and geographic blocks in each fold
print(data.groupby("fold_id").agg(
    n_samples=("plot_id", "size"),
    n_blocks=("block_id", "nunique")
))

# Distribution of target classes across folds
print(pd.crosstab(
    data["habitat_type"],
    data["fold_id"]
))

         n_samples  n_blocks
fold_id                     
0              337        56
1              328        68
2              327        68
3              339        70
4              340        59
fold_id        0   1   2   3   4
habitat_type                    
0             27  37  34  35  34
1630           0   0   0   1   0
2180          21  17   9  29  20
5130           3   0   0   1   0
6280           2   0   0   0   0
6430           0   0   0   0   1
6510           0   0   0   1   0
6530           2   2   2   2   2
7110           0   2   1   1   1
7120           2   2   0   0   3
7140           1   0   0   0   0
7220           0   0   0   0   1
8210           0   0   1   0   0
9010*         64  47  48  72  68
9020*          8  18  22  27  28
9050          15  20  35  24  36
9060          13  21  33  30  20
9070          19  20  18  28  12
9080*         41  45  31  25  35
9180*          3  23  12  16  26
91D0*         30  24  41  22  35
91E0*         52  23  13   9   7
91F0 

In [21]:
# Only observations eligible for modelling
data = monitoring_centres[
    monitoring_centres["split"].isin(["train_cv", "test"])
]

# Number of observations and geographic blocks in each fold
print(data.groupby("fold_id").agg(
    n_samples=("plot_id", "size"),
    n_blocks=("block_id", "nunique")
))

# Distribution of target classes across folds
print(pd.crosstab(
    data["overall_conservation_value"],
    data["fold_id"]
))

         n_samples  n_blocks
fold_id                     
0              337        56
1              328        68
2              327        68
3              339        70
4              340        59
fold_id                       0    1    2    3    4
overall_conservation_value                         
A                           166  153  143  162  151
B                            81   69   90   86   85
C                            51   58   54   47   60


These folds were stratified based on structural preservation, so it is clear that other classifications representability is not as balanced across folds. So folding would need to be done again for other features if needed. Also we have probably not enough data without inventories, so inventory data might be needed. Or bigger folds created.

In [22]:
print(data.groupby("block_id")["fold_id"].nunique().value_counts())
print(monitoring_centres["split"].value_counts())
print(pd.crosstab(
    data["inventory_year"],
    data["fold_id"]
))

fold_id
1    321
Name: count, dtype: int64
split
train_cv            1334
test                 337
buffer_excluded       15
overlap_excluded      12
Name: count, dtype: int64
fold_id          0   1   2   3   4
inventory_year                    
2017            20  24  31  34  30
2018            20   4   7   6   6
2019            21  15  17  31  39
2020            24  33  44  35  42
2021            38  62  32  39  43
2022            20  51  17  62  36
2023            77  17  34  37  34
2024            40  42  44  40  30
2025            31  37  43  28  58
2026            46  43  58  27  22


It casn be seen that there are not many exclusions, but further exploration is needed to be sure that close monitoring plots would not fall into separate folds.

![Fold distribution](<Example_fold distribution.png>)

## 11. What to do in the NEXT notebook

- Select a **single target variable** to model, such as
  `structural_preservation` (recommended first baseline) or `habitat_type`.
- Extract predictor summaries inside the **20 m monitoring circle**, and
  optionally within larger surrounding raster neighbourhoods.
- Keep records only where the chosen target and needed raster features exist.
- **Never use** rows with `split == 'test'`, `buffer_excluded`, or
  `overlap_excluded` during hyperparameter selection.
- For folds 1–4, use `buffered_cv_indices()` to create rotating validation
  sets. On a DataFrame restricted to eligible rows, call
  `reset_index(drop=True)` first, so the returned indices match model rows.
- Choose parameters using the four validation scores. Refit the selected
  model on all eligible folds 1–4, then evaluate **once** on test fold 0.

**Important scientific limitations:**

- Polygon-derived points inherit whole-polygon assessments. They are weak
  labels, not independent proof of the condition at each point.
- If you choose `lk_seisund`, remember the monitoring CSV has no direct
  equivalent; those labels are available only from inventory polygons.
- Geographical CV tests transfer to new monitored locations. It does not by
  itself establish accuracy in unmonitored, non-protected forests.
- Check whether the measured 20 m footprint exactly matches the monitoring
  protocol. Both the fold buffer and polygon match currently use centres.
